# 第 1 章 · 问题定义

## 本节目标

定义清楚"我们在解决什么问题"，而不是上来就 import sklearn。

## 背景

1912 年 4 月 15 日，泰坦尼克号在首航中撞上冰山沉没。船上 2224 名乘客和船员中，约 1502 人遇难，仅 722 人生还。

事故调查发现，救生艇数量不足是造成高死亡率的重要原因之一——但救生艇的使用并非完全随机。当时执行的"妇女儿童优先"原则，以及乘客的舱位等级、国籍等因素，都在不同程度上影响着一个人的幸存机会。

## 任务

给定部分乘客的个人信息（姓名、性别、年龄、舱位等级、票价等），预测该乘客是否幸存。这是一个**二分类问题**：Survived = 1（幸存）或 0（遇难）。

## 评估指标

- **Accuracy（准确率）**：竞赛使用的官方指标，即预测正确的比例
- **混淆矩阵**：理解模型在"预测幸存但实际遇难"和"预测遇难但实际幸存"上的表现——在真实场景中，这两种错误的代价不同

## 面试钩子

> 如果我是一个航运安全调查员，知道了"谁更容易幸存"，我就能反推灾难中什么因素最关键——阶级？性别？年龄？还是纯粹的运气？

带着这个问题，开始我们的分析。

In [ ]:
# === 第 1 章：导入库 & 加载数据 ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示（Windows 常用字体）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
sns.set_palette('Set2')

# 加载数据
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f'训练集: {train.shape[0]} 条, {train.shape[1]} 列')
print(f'测试集: {test.shape[0]} 条, {test.shape[1]} 列')
print(f'\n训练集列名: {list(train.columns)}')

In [ ]:
# === 环境自检：自动安装缺失依赖 ===
import subprocess, sys, importlib

required = {
    'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib',
    'seaborn': 'seaborn', 'sklearn': 'scikit-learn',
    'xgboost': 'xgboost', 'shap': 'shap', 'missingno': 'missingno',
}

missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]

if missing:
    print(f'检测到缺失依赖: {missing}，正在自动安装...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
    print('安装完成！请重启 kernel（Kernel → Restart）后重新运行所有 cell。')
else:
    print('所有依赖已就绪')

# 第 2 章 · 全局概览 & 探索性数据分析（EDA）

## 本节目标

在建模之前，先**把数据"看透"**。EDA 不是无脑画图——每张图都应该回答一个具体问题。我的节奏是：**提出假设 → 用图验证 → 得出阶段性结论**。

## 2.1 数据初探：结构、类型、缺失值

先上一张缺失值矩阵图——这比 `df.isnull().sum()` 直观得多，一眼就能看到数据的"窟窿"在哪里。

In [ ]:
# === 2.1 数据初探 ===
print("=" * 50)
print("数据结构概览")
print("=" * 50)
train.info()

print("\n" + "=" * 50)
print("数值特征统计描述")
print("=" * 50)
display(train.describe().T)

print("\n" + "=" * 50)
print("缺失值统计")
print("=" * 50)
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(1)
missing_df = pd.DataFrame({'缺失数': missing, '缺失比例(%)': missing_pct})
display(missing_df[missing_df['缺失数'] > 0].sort_values('缺失数', ascending=False))

# 缺失值矩阵图
plt.figure(figsize=(12, 6))
msno.matrix(train)
plt.title('训练集缺失值矩阵图', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## 2.2 单变量分析：逐一验证假设

数据概况看完了。现在逐个特征问：**它和"是否幸存"有没有关系？关系多大？方向是怎样的？**

In [ ]:
# === 2.2 目标变量：Survived 分布 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 饼图
survived_counts = train['Survived'].value_counts()
axes[0].pie(survived_counts, labels=['遇难 (0)', '幸存 (1)'], autopct='%1.1f%%',
            colors=['#E74C3C', '#2ECC71'], explode=(0, 0.05), startangle=90)
axes[0].set_title('幸存 vs 遇难 比例', fontsize=13)

# 柱状图
bars = axes[1].bar(['遇难', '幸存'], survived_counts.values, color=['#E74C3C', '#2ECC71'])
axes[1].set_title('幸存 vs 遇难 人数', fontsize=13)
for bar, val in zip(bars, survived_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
                 ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'幸存率: {train["Survived"].mean()*100:.1f}% — 不到 4 成的人活下来了。这就是泰坦尼克号。')

### 🚻 Sex：性别

**假设：** "妇女儿童优先"不是说说而已——女性的幸存率应该远高于男性。

**验证：**

In [ ]:
# === 2.2 Sex vs Survived ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：男女幸存率
sex_survived = train.groupby('Sex')['Survived'].mean() * 100
bars = axes[0].bar(['女性', '男性'], sex_survived.values, color=['#E91E63', '#2196F3'])
axes[0].set_title('不同性别的幸存率', fontsize=13)
axes[0].set_ylabel('幸存率 (%)')
for bar, val in zip(bars, sex_survived.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=14, fontweight='bold')

# 右图：性别 x 幸存人数堆叠图
sex_survived_count = pd.crosstab(train['Sex'], train['Survived'])
sex_survived_count.columns = ['遇难', '幸存']
sex_survived_count.index = ['女性', '男性']
sex_survived_count.plot(kind='bar', stacked=True, ax=axes[1],
                        color=['#E74C3C', '#2ECC71'])
axes[1].set_title('性别 × 幸存人数', fontsize=13)
axes[1].set_ylabel('人数')
axes[1].legend(loc='upper right')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f'女性幸存率: {sex_survived["female"]:.1f}%')
print(f'男性幸存率: {sex_survived["male"]:.1f}%')
print(f'差异: {sex_survived["female"] - sex_survived["male"]:.1f} 个百分点')
print(f'\n> 结论: 女性幸存率是男性的 {sex_survived["female"]/sex_survived["male"]:.1f} 倍。"妇女优先"不是传言。')

### 🎫 Pclass：舱位等级

**假设：** 富人的命更值钱——头等舱乘客幸存率最高，三等舱最低。

In [ ]:
# === 2.2 Pclass vs Survived ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pclass_survived = train.groupby('Pclass')['Survived'].mean() * 100
bars = axes[0].bar(['头等舱 (1)', '二等舱 (2)', '三等舱 (3)'],
                   pclass_survived.values, color=['#FFD700', '#C0C0C0', '#CD7F32'])
axes[0].set_title('不同舱位等级的幸存率', fontsize=13)
axes[0].set_ylabel('幸存率 (%)')
for bar, val in zip(bars, pclass_survived.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=14, fontweight='bold')

pclass_count = pd.crosstab(train['Pclass'], train['Survived'])
pclass_count.columns = ['遇难', '幸存']
pclass_count.plot(kind='bar', stacked=True, ax=axes[1], color=['#E74C3C', '#2ECC71'])
axes[1].set_title('舱位等级 × 幸存人数', fontsize=13)
axes[1].set_ylabel('人数')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()
print(f'> 结论: 头等舱幸存率 63% > 二等舱 47% > 三等舱 24%。财富直接影响了生存机会。')

### 👶 Age：年龄

**假设：** 小孩优先上救生艇——年龄越小，幸存率越高。

In [ ]:
# === 2.2 Age vs Survived ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 年龄分布（按幸存状态）
for survived, label, color in [(0, '遇难', '#E74C3C'), (1, '幸存', '#2ECC71')]:
    subset = train[train['Survived'] == survived]['Age'].dropna()
    axes[0].hist(subset, bins=30, alpha=0.6, label=label, color=color, edgecolor='white')

axes[0].set_title('年龄分布（按幸存状态）', fontsize=13)
axes[0].set_xlabel('年龄')
axes[0].set_ylabel('人数')
axes[0].legend()

# 年龄分段幸存率
age_bins = [0, 12, 18, 30, 50, 80]
age_labels = ['儿童(0-12)', '青少年(13-18)', '青年(19-30)', '中年(31-50)', '老年(51+)']
train['AgeGroup_temp'] = pd.cut(train['Age'], bins=age_bins, labels=age_labels)
age_survived = train.groupby('AgeGroup_temp', observed=False)['Survived'].mean() * 100

bars = axes[1].bar(age_labels, age_survived.values,
                   color=['#3498DB', '#2ECC71', '#F39C12', '#E74C3C', '#9B59B6'])
axes[1].set_title('各年龄段幸存率', fontsize=13)
axes[1].set_ylabel('幸存率 (%)')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, age_survived.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()
train.drop('AgeGroup_temp', axis=1, inplace=True)
print(f'> 结论: 儿童(0-12岁)幸存率最高({age_survived.iloc[0]:.1f}%)，但并非年龄越小越好——"小孩优先"主要作用于低龄儿童。')

### 👨‍👩‍👧‍👦 SibSp & Parch：家庭同行人数

**假设：** 有家人一起的乘客幸存率更高（互相帮助），但太多人又互相拖累。

In [ ]:
# === 2.2 SibSp & Parch vs Survived ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, ['SibSp', 'Parch'],
                          ['兄弟姐妹/配偶数', '父母/子女数']):
    sib_survived = train.groupby(col)['Survived'].mean() * 100
    counts = train.groupby(col).size()
    bars = ax.bar(sib_survived.index.astype(str), sib_survived.values,
                  color=plt.cm.RdYlGn(sib_survived.values / 100))
    ax.set_title(f'{title} vs 幸存率', fontsize=13)
    ax.set_ylabel('幸存率 (%)')
    # 标注每组的样本量
    for i, (bar, val, cnt) in enumerate(zip(bars, sib_survived.values, counts.values)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{val:.1f}%\n(n={cnt})', ha='center', fontsize=9)

plt.tight_layout()
plt.show()
print('> 观察: SibSp=1~2 时幸存率最高；SibSp=0 或 ≥3 时幸存率下降。家庭规模有"最优区间"。')

### 💰 Fare：票价 & 🚢 Embarked：登船港口

In [ ]:
# === 2.2 Fare & Embarked vs Survived ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fare 分布
for survived, label, color in [(0, '遇难', '#E74C3C'), (1, '幸存', '#2ECC71')]:
    subset = train[train['Survived'] == survived]['Fare']
    axes[0, 0].hist(subset, bins=40, alpha=0.6, label=label, color=color, edgecolor='white', range=(0, 150))
axes[0, 0].set_title('票价分布（按幸存状态，截断至150）', fontsize=13)
axes[0, 0].set_xlabel('票价')
axes[0, 0].legend()

# Fare 分箱
train['FareBin_temp'] = pd.qcut(train['Fare'], 4, labels=['低', '中低', '中高', '高'])
fare_survived = train.groupby('FareBin_temp', observed=False)['Survived'].mean() * 100
bars = axes[0, 1].bar(fare_survived.index, fare_survived.values,
                      color=['#3498DB', '#2ECC71', '#F39C12', '#E74C3C'])
axes[0, 1].set_title('票价四分位 vs 幸存率', fontsize=13)
axes[0, 1].set_ylabel('幸存率 (%)')
for bar, val in zip(bars, fare_survived.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%', ha='center', fontweight='bold')
train.drop('FareBin_temp', axis=1, inplace=True)

# Embarked
embarked_survived = train.groupby('Embarked')['Survived'].mean() * 100
embarked_count = train['Embarked'].value_counts()
colors_emb = ['#3498DB', '#E74C3C', '#2ECC71']
bars = axes[1, 0].bar(['S (南安普顿)', 'C (瑟堡)', 'Q (皇后镇)'], embarked_survived.values,
                      color=colors_emb)
axes[1, 0].set_title('登船港口 vs 幸存率', fontsize=13)
axes[1, 0].set_ylabel('幸存率 (%)')
for bar, val, cnt in zip(bars, embarked_survived.values, embarked_count.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1f}%\n(n={cnt})', ha='center', fontweight='bold')

# 相关矩阵热力图
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr = train[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            square=True, ax=axes[1, 1])
axes[1, 1].set_title('数值特征相关性热力图', fontsize=13)

plt.tight_layout()
plt.show()

print(f'> 结论: 票价越高幸存率越高（和 Pclass 相关），C 港登船者幸存率最高（头等舱乘客多在 C 港登船）。')

## 2.3 交叉分析：单变量看不出的隐藏模式

单变量分析告诉我们每个特征单独的作用。但特征之间会**互相作用**——比如三等舱的女性 vs 头等舱的女性，她们的待遇一样吗？

In [ ]:
# === 2.3 交叉分析 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sex × Pclass
pivot1 = train.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean') * 100
sns.heatmap(pivot1, annot=True, fmt='.1f', cmap='RdYlGn', center=50,
            ax=axes[0], cbar_kws={'label': '幸存率 (%)'})
axes[0].set_title('Sex × Pclass 幸存率热力图', fontsize=13)
axes[0].set_xlabel('舱位等级')
axes[0].set_yticklabels(['女性', '男性'], rotation=0)

# Age × Sex
train['AgeBin_temp'] = pd.cut(train['Age'], bins=[0, 18, 50, 80], labels=['儿童', '成人', '老年'])
pivot2 = train.pivot_table(values='Survived', index='AgeBin_temp', columns='Sex',
                           aggfunc='mean', observed=False) * 100
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='RdYlGn', center=50,
            ax=axes[1], cbar_kws={'label': '幸存率 (%)'})
axes[1].set_title('Age × Sex 幸存率热力图', fontsize=13)
train.drop('AgeBin_temp', axis=1, inplace=True)

plt.tight_layout()
plt.show()

print('> 交叉分析的发现:')
print('  1. 头等舱女性幸存率 97%——几乎全活。三等舱男性幸存率仅 14%——几乎全死。')
print('  2. 在头等舱内部，性别差距接近 60 个百分点；在三等舱内部，差距缩小到约 30 个百分点。')
print('  3. 男性儿童幸存率远高于男性成人（"男孩是孩子，成年男人不是"），但女性各年龄段差异不大。')

### EDA 阶段性总结

| 特征 | 与 Survived 的关系 | 关键数字 |
|------|-------------------|----------|
| Sex | **最强单一预测因子** | 女 74% vs 男 19% |
| Pclass | 强正相关，财富决定命运 | 头等 63% → 三等 24% |
| Age | 儿童优先，但非线性 | 儿童 54%，成人/老人 ~38% |
| Fare | 与 Pclass 高度共线 | 高票价 = 高幸存率 |
| SibSp/Parch | 中等家庭规模最优 | 1-2人同行幸存率最高 |
| Embarked | 弱相关，受 Pclass 混杂 | C 港最高因头等舱乘客多 |

> **对建模的启示：** Sex 和 Pclass 是核心特征。Fare 和 Pclass 共线，建模时需注意。Age 的处理方式会显著影响模型效果。

# 第 3 章 · 缺失值处理

## 本节目标

缺失值处理不是 `df.fillna(df.median())` 一行搞定的事。每个特征缺失的原因不同，处理方式也应该不同。**关键不是"填了什么"，而是"为什么这样填"。**

## 3.1 Age（177 个缺失，19.9%）

**为什么不能直接填中位数？** 因为 Age 的缺失可能不是完全随机的——缺失年龄的人和记录年龄的人，在其他特征上可能有系统性差异。先验证这一点。

**决策思路：**
1. 分析 Age 缺失是否和 Pclass/Sex 相关
2. 如果相关 → 用分组统计量填充
3. 如果不相关 → 用全局统计量填充
4. 填充后验证分布是否改变

In [ ]:
# === 3.1 Age：分析缺失模式 ===
# 先标记 Age 是否缺失
train['AgeMissing'] = train['Age'].isnull().astype(int)
test['AgeMissing'] = test['Age'].isnull().astype(int)

# 检查 Age 缺失是否和 Pclass 相关
print('=== Age 缺失比例（按 Pclass） ===')
age_missing_by_pclass = train.groupby('Pclass')['AgeMissing'].mean() * 100
print(age_missing_by_pclass.to_string())
print(f'\n三等舱缺失比例最高({age_missing_by_pclass[3]:.1f}%)，头等舱最低({age_missing_by_pclass[1]:.1f}%)。')

# 检查 Age 缺失和 Sex 的关系
print('\n=== Age 缺失比例（按 Sex） ===')
age_missing_by_sex = train.groupby('Sex')['AgeMissing'].mean() * 100
print(age_missing_by_sex.to_string())
print(f'\n差异不大，性别不是主因。')

# 画图：Age 缺失 vs Pclass
fig, ax = plt.subplots(figsize=(8, 4))
train.groupby('Pclass')['AgeMissing'].mean().plot(kind='bar', ax=ax, color=['#FFD700', '#C0C0C0', '#CD7F32'])
ax.set_title('各舱位等级的 Age 缺失比例', fontsize=13)
ax.set_ylabel('缺失比例')
ax.set_xlabel('舱位等级')
ax.set_xticklabels(['头等舱', '二等舱', '三等舱'], rotation=0)
plt.tight_layout()
plt.show()

print('\n> 结论: Age 缺失不是随机的——和 Pclass 有关。简单的全局中位数填充会忽略这个结构。')

### Title 提取 + Age 填充

> **为什么用 Title？** Name 中的称谓（Mr/Mrs/Miss/Master 等）同时包含了性别和身份信息。Master 是未成年男性专用的称谓，所以 Title 和 Age 高度相关——比单纯用 Pclass 分组更精确。

In [ ]:
# === 3.1 Title 提取 & Age 填充 ===
import re

def extract_title(name):
    """从 Name 中提取称谓"""
    match = re.search(r',\s*([^\.]+)\.', name)
    if match:
        return match.group(1).strip()
    return 'Other'

# 合并 train 和 test 统一处理
all_data = pd.concat([train, test], axis=0, sort=False)

all_data['Title'] = all_data['Name'].apply(extract_title)

# 将稀有 Title 归并
title_map = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Royal', 'Sir': 'Royal', 'the Countess': 'Royal',
    'Jonkheer': 'Royal', 'Don': 'Royal', 'Dona': 'Royal',
    'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Dr': 'Officer', 'Rev': 'Officer'
}
all_data['Title'] = all_data['Title'].replace(title_map)

print('=== Title 分布 ===')
print(all_data['Title'].value_counts())

# 看 Title 和 Age 的关系
print('\n=== 各 Title 的 Age 中位数 ===')
title_age_median = all_data.groupby('Title')['Age'].median().sort_values()
print(title_age_median)

# 用 Title 分组中位数填充 Age
for title in all_data['Title'].unique():
    median_age = all_data.loc[all_data['Title'] == title, 'Age'].median()
    mask = (all_data['Title'] == title) & (all_data['Age'].isnull())
    all_data.loc[mask, 'Age'] = median_age

print(f'\nAge 填充完成。剩余缺失: {all_data["Age"].isnull().sum()} 条')

In [ ]:
# === 3.1 Age 填充前后分布对比 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 填充前的分布
original_age = pd.concat([train['Age'].dropna(), test['Age'].dropna()])
axes[0].hist(original_age, bins=30, color='#E74C3C', alpha=0.7, edgecolor='white')
axes[0].axvline(original_age.median(), color='black', linestyle='--', linewidth=2, label=f'中位数: {original_age.median():.1f}')
axes[0].set_title('Age 填充前分布', fontsize=13)
axes[0].legend()

# 填充后的分布
axes[1].hist(all_data['Age'], bins=30, color='#2ECC71', alpha=0.7, edgecolor='white')
axes[1].axvline(all_data['Age'].median(), color='black', linestyle='--', linewidth=2, label=f'中位数: {all_data["Age"].median():.1f}')
axes[1].set_title('Age 填充后分布（Title 分组中位数）', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'填充前中位数: {original_age.median():.1f} → 填充后中位数: {all_data["Age"].median():.1f}')
print('> 分布形态基本一致，说明填充没有引入明显偏差。')

## 3.2 Embarked（2 个缺失，0.2%）& Fare（1 个缺失，0.1%）

这两个特征的缺失极少，填充策略简单直接。

In [ ]:
# === 3.2 Embarked & Fare 填充 ===
# Embarked: 检查缺失的两个人
embarked_missing = all_data[all_data['Embarked'].isnull()]
print('=== Embarked 缺失的乘客信息 ===')
display(embarked_missing[['Pclass', 'Fare', 'Sex', 'Cabin']])
print('\n两人都是头等舱、票价$80、女性。同舱位+票价段中，S港人数最多。填充 S。')

all_data['Embarked'] = all_data['Embarked'].fillna('S')

# Fare: 测试集中有 1 条缺失
fare_missing = all_data[all_data['Fare'].isnull()]
if len(fare_missing) > 0:
    print('\n=== Fare 缺失的乘客信息 ===')
    display(fare_missing[['Pclass', 'Sex', 'Age', 'Embarked']])
    fare_median = all_data.loc[all_data['Pclass'] == fare_missing['Pclass'].values[0], 'Fare'].median()
    all_data.loc[all_data['Fare'].isnull(), 'Fare'] = fare_median
    print(f'\n用 Pclass 中位数填充 Fare: {fare_median:.2f}')

print(f'\nEmbarked 缺失: {all_data["Embarked"].isnull().sum()}, Fare 缺失: {all_data["Fare"].isnull().sum()}')

## 3.3 Cabin（687 个缺失，77.1%）

> **为什么不能直接删？** Cabin 缺失 77%，直接删看似最省事。但 Cabin 的**首字母代表甲板层**，而甲板层和舱位等级有关。此外，**"有没有 Cabin 记录"本身可能是个信号**——有记录的乘客信息更完整，可能意味着不同待遇。

**决策：** 不填缺失值，而是构造两个新特征：
- `Cabin_deck`：Cabin 首字母（甲板层），缺失标 'U'（Unknown）
- `HasCabin`：是否记录了 Cabin

In [ ]:
# === 3.3 Cabin：提取甲板层 & 是否有舱位记录 ===
all_data['Cabin_deck'] = all_data['Cabin'].apply(
    lambda x: str(x)[0] if pd.notnull(x) else 'U'
)
all_data['HasCabin'] = all_data['Cabin'].notnull().astype(int)

# 验证：甲板层和 Pclass 的关系
print('=== 各甲板层的 Pclass 分布 ===')
deck_pclass = all_data.groupby('Cabin_deck')['Pclass'].value_counts().unstack(fill_value=0)
print(deck_pclass.to_string())

# 验证：HasCabin 和 Survived 的关系（仅训练集）
train_has_cabin = all_data[:len(train)][['HasCabin', 'Survived']]
has_cabin_rate = train_has_cabin.groupby('HasCabin')['Survived'].mean() * 100
print(f'\n有 Cabin 记录的幸存率: {has_cabin_rate.get(1, 0):.1f}%')
print(f'无 Cabin 记录的幸存率: {has_cabin_rate.get(0, 0):.1f}%')
print('> 有 Cabin 记录的人幸存率明显更高——这个特征值得保留。')

### 缺失值处理总结

| 特征 | 缺失数 | 处理方式 | 核心理由 |
|------|--------|----------|----------|
| Age | 177 | Title 分组中位数填充 | Title 比 Pclass 更精准地关联年龄 |
| Embarked | 2 | 填充众数 'S' | 极少缺失 + 业务推断 |
| Cabin | 687 | 提取 Cabin_deck + HasCabin | 缺失太多不能填，但信息不能浪费 |
| Fare | 1 | Pclass 中位数填充 | 票价与舱位强相关 |

现在 `all_data` 中没有基础缺失值了，可以进入特征工程阶段。

# 第 4 章 · 特征工程

## 本节目标

原始特征只是"原材料"。好的特征是**从数据里挖出来的新信息**——不是一个 `pd.get_dummies()` 能解决的。每个新特征都要回答：**它带来了什么原始特征没有的信息？**

In [ ]:
# === 第 4 章：特征工程 ===

# 4.1 FamilySize & IsAlone
all_data['FamilySize'] = all_data['SibSp'] + all_data['Parch'] + 1
all_data['IsAlone'] = (all_data['FamilySize'] == 1).astype(int)

# 验证
train_len = len(train)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fs_survived = all_data[:train_len].groupby('FamilySize')['Survived'].mean() * 100
fs_counts = all_data[:train_len].groupby('FamilySize').size()
bars = axes[0].bar(fs_survived.index.astype(str), fs_survived.values,
                   color=['#E74C3C' if v < 40 else '#2ECC71' for v in fs_survived.values])
axes[0].set_title('FamilySize vs 幸存率', fontsize=13)
axes[0].set_ylabel('幸存率 (%)')
for bar, val, cnt in zip(bars, fs_survived.values, fs_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%\n(n={cnt})', ha='center', fontsize=9)

alone_survived = all_data[:train_len].groupby('IsAlone')['Survived'].mean() * 100
bars = axes[1].bar(['有家人同行', '独自出行'], alone_survived.values,
                   color=['#2ECC71', '#E74C3C'])
axes[1].set_title('IsAlone vs 幸存率', fontsize=13)
axes[1].set_ylabel('幸存率 (%)')
for bar, val in zip(bars, alone_survived.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print(f'FamilySize=2~4 时幸存率最高。独自出行的幸存率({alone_survived[1]:.1f}%)明显低于有家人同行({alone_survived[0]:.1f}%)。')

In [ ]:
# === 4.2 AgeBin & FareBin ===
# AgeBin: 基于 EDA 发现的年龄效应（非线性的）
all_data['AgeBin'] = pd.cut(all_data['Age'], bins=[0, 12, 18, 30, 50, 80],
                            labels=[0, 1, 2, 3, 4]).astype(int)

# FareBin: qcut 处理长尾
all_data['FareBin'] = pd.qcut(all_data['Fare'], 4, labels=[0, 1, 2, 3]).astype(int)

# 验证
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

agebin_survived = all_data[:train_len].groupby('AgeBin')['Survived'].mean() * 100
axes[0].bar(['儿童', '青少年', '青年', '中年', '老年'], agebin_survived.values,
            color=['#3498DB', '#2ECC71', '#F39C12', '#E74C3C', '#9B59B6'])
axes[0].set_title('AgeBin vs 幸存率', fontsize=13)
axes[0].set_ylabel('幸存率 (%)')

farebin_survived = all_data[:train_len].groupby('FareBin')['Survived'].mean() * 100
axes[1].bar(['低', '中低', '中高', '高'], farebin_survived.values,
            color=['#3498DB', '#2ECC71', '#F39C12', '#E74C3C'])
axes[1].set_title('FareBin vs 幸存率', fontsize=13)
axes[1].set_ylabel('幸存率 (%)')

plt.tight_layout()
plt.show()

In [ ]:
# === 4.3 TicketGroup：同一票号的人数 ===
ticket_counts = all_data['Ticket'].value_counts()
all_data['TicketGroup'] = all_data['Ticket'].map(ticket_counts)

# 验证
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tg_survived = all_data[:train_len].groupby('TicketGroup')['Survived'].mean() * 100
tg_counts = all_data[:train_len].groupby('TicketGroup').size()
# 展示 1-6 人团体（6+ 归并）
tg_display = tg_survived.loc[1:6]
tg_n = tg_counts.loc[1:6]
bars = axes[0].bar(tg_display.index.astype(str), tg_display.values,
                   color=plt.cm.RdYlGn(np.array(tg_display.values) / 100))
axes[0].set_title('TicketGroup（同票号人数）vs 幸存率', fontsize=13)
for bar, val, cnt in zip(bars, tg_display.values, tg_n.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%\n(n={cnt})', ha='center', fontsize=9)

# TicketGroup 和 FamilySize 的关系
axes[1].scatter(all_data['FamilySize'], all_data['TicketGroup'], alpha=0.3, c='#3498DB')
axes[1].set_xlabel('FamilySize')
axes[1].set_ylabel('TicketGroup（同票号人数）')
axes[1].set_title('TicketGroup vs FamilySize', fontsize=13)
# 加 y=x 参考线
max_val = max(all_data['FamilySize'].max(), all_data['TicketGroup'].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='y=x')
axes[1].legend()

plt.tight_layout()
plt.show()
print('> TicketGroup 和 FamilySize 不完全一致——有的团体共用票号但并非家人。可以互补。')

### 特征工程总结

| 新特征 | 来源 | 带来的新信息 |
|--------|------|-------------|
| Title | Name 正则提取 | 身份 + 性别 + 年龄的综合信号 |
| FamilySize | SibSp + Parch + 1 | 家庭规模，捕捉最优区间 |
| IsAlone | FamilySize == 1 | 独行 vs 有伴的二元差异 |
| AgeBin | Age 分段 | 年龄的非线性效应（小孩 vs 成人待遇不同） |
| FareBin | Fare 四分位 | 票价长尾的离散化 |
| Cabin_deck | Cabin 首字母 | 甲板层 = Pclass 的补充维度 |
| HasCabin | Cabin 是否缺失 | 信息完整性 = 乘客地位信号 |
| TicketGroup | Ticket 票号计数 | 团体出行（可和 FamilySize 互补） |
| AgeMissing | Age 是否缺失 | 缺失本身可能是信号 |

> 现在有 9 个新特征 + 原始特征（选 Sex/Pclass/SibSp/Parch/Fare/Embarked）。接下来进入建模阶段。

# 第 5 章 · 建模 & 对比

## 本节目标

不是"哪一个模型跑得准就用哪一个"。而是**系统性地对比 5 个模型，理解谁好谁坏、为什么**，然后基于数据和理解选择最优方案。

In [ ]:
# === 第 5 章：数据准备 ===
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 选特征
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
            'Title', 'FamilySize', 'IsAlone', 'AgeBin', 'FareBin',
            'Cabin_deck', 'HasCabin', 'TicketGroup', 'AgeMissing']

# Label Encode 分类特征
le_cols = ['Sex', 'Embarked', 'Title', 'Cabin_deck']
for col in le_cols:
    all_data[col] = LabelEncoder().fit_transform(all_data[col].astype(str))

# 分离 train/test
train_processed = all_data[:train_len].copy()
test_processed = all_data[train_len:].copy()

X = train_processed[features]
y = train_processed['Survived'].astype(int)
X_test_final = test_processed[features]

# 标准化（对 SVM/KNN/Logistic 重要）
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test_final)

print(f'训练特征: {X.shape[1]} 维, 样本: {X.shape[0]} 条')
print(f'测试样本: {X_test_final.shape[0]} 条')
print(f'特征列表: {features}')

## 5.1 第一轮：Baseline 对比（默认参数，5-fold CV）

**实验设计：** 5 个模型都用默认参数跑 5 折交叉验证，记录 Accuracy ± std。目的是**在不调参的前提下看谁天然适合这个数据**。

In [ ]:
# === 5.1 Baseline 模型对比 ===
import time

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42, verbosity=0)
}

results = []
for name, model in models.items():
    start = time.time()
    # KNN/SVM 用 scaled 数据，树模型用原始数据
    if 'KNN' in name or 'SVM' in name or 'Logistic' in name:
        scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    else:
        scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    elapsed = time.time() - start
    results.append({
        '模型': name,
        'CV Accuracy': f'{scores.mean():.4f}',
        'Std': f'{scores.std():.4f}',
        '最高': f'{scores.max():.4f}',
        '最低': f'{scores.min():.4f}',
        '训练时间(s)': f'{elapsed:.3f}'
    })

results_df = pd.DataFrame(results).sort_values('CV Accuracy', ascending=False)
display(results_df)

print(f'\n最佳 baseline: {results_df.iloc[0]["模型"]} (CV Accuracy = {results_df.iloc[0]["CV Accuracy"]})')

## 5.2 第二轮：Top 模型调参（GridSearchCV）

> **为什么手动 GridSearchCV 而不是 Optuna/RandomSearch？** 对于这种小数据集，搜索空间不大，手动 GridSearch 可以让面试官清楚地看到"你试了什么参数、最优组合是什么"。可解释性 > 效率。

In [ ]:
# === 5.2 GridSearchCV 调参 ===

# --- Random Forest ---
print('=== Random Forest 调参 ===')
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params,
                       cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X, y)
print(f'最优参数: {rf_grid.best_params_}')
print(f'最优 CV Accuracy: {rf_grid.best_score_:.4f}')

# --- XGBoost ---
print('\n=== XGBoost 调参 ===')
xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}
xgb_grid = GridSearchCV(XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0),
                        xgb_params, cv=5, scoring='accuracy', n_jobs=-1)
xgb_grid.fit(X, y)
print(f'最优参数: {xgb_grid.best_params_}')
print(f'最优 CV Accuracy: {xgb_grid.best_score_:.4f}')

# --- Logistic Regression ---
print('\n=== Logistic Regression 调参 ===')
lr_params = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}
lr_grid = GridSearchCV(LogisticRegression(max_iter=2000, random_state=42), lr_params,
                       cv=5, scoring='accuracy')
lr_grid.fit(X_scaled, y)
print(f'最优参数: {lr_grid.best_params_}')
print(f'最优 CV Accuracy: {lr_grid.best_score_:.4f}')

# 汇总
print('\n=== 调参后对比 ===')
tuned_results = pd.DataFrame([
    {'模型': 'Logistic Regression (调参)', 'CV Accuracy': f'{lr_grid.best_score_:.4f}'},
    {'模型': 'Random Forest (调参)', 'CV Accuracy': f'{rf_grid.best_score_:.4f}'},
    {'模型': 'XGBoost (调参)', 'CV Accuracy': f'{xgb_grid.best_score_:.4f}'},
])
display(tuned_results)

# 选择最佳模型
best_model = rf_grid.best_estimator_ if rf_grid.best_score_ >= max(xgb_grid.best_score_, lr_grid.best_score_) else \
             (xgb_grid.best_estimator_ if xgb_grid.best_score_ >= lr_grid.best_score_ else lr_grid.best_estimator_)
print(f'\n最终选用: {type(best_model).__name__}')

# 第 6 章 · 模型解释 & 业务结论

## 本节目标

准确率是一个数字。业务价值是理解**"为什么这个人幸存"和"这个因素有多大影响"**。这是面试官合上 Notebook 后还记得的东西。

In [ ]:
# === 6.1 特征重要性 ===
# 训练最佳模型
best_model.fit(X, y)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 如果最佳模型有 feature_importances_
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-10:]  # top 10
    axes[0].barh(range(len(indices)), importances[indices], color='#3498DB')
    axes[0].set_yticks(range(len(indices)))
    axes[0].set_yticklabels([features[i] for i in indices])
    axes[0].set_title(f'{type(best_model).__name__} 特征重要性 (Top 10)', fontsize=13)
    axes[0].set_xlabel('重要性')

# 也画一个 LR 的系数（如果有）
if 'lr_grid' in dir():
    lr_coef = abs(lr_grid.best_estimator_.coef_[0])
    lr_indices = np.argsort(lr_coef)[-10:]
    axes[1].barh(range(len(lr_indices)), lr_coef[lr_indices], color='#2ECC71')
    axes[1].set_yticks(range(len(lr_indices)))
    axes[1].set_yticklabels([features[i] for i in lr_indices])
    axes[1].set_title('Logistic Regression 系数绝对值 (Top 10)', fontsize=13)

plt.tight_layout()
plt.show()

print('> 无论用哪个模型，Sex 和 Title 始终是最重要的特征——性别是泰坦尼克号上决定生死的头号因素。')
print('> Pclass 和 Fare 紧随其后——财富阶级是第二因素。')

In [ ]:
# === 6.2 SHAP 分析 ===
import shap

# 用 TreeExplainer（最快，适用于 RF/XGBoost）
if hasattr(best_model, 'feature_importances_'):
    explainer = shap.TreeExplainer(best_model)
    # 取训练集的一个子集构建解释（节省内存）
    X_sample = X.iloc[:200]

    # shap 0.46+ 推荐使用 explainer(X) 而非 explainer.shap_values()
    try:
        shap_values = explainer(X_sample)
        # 二分类时取正类
        if len(shap_values.values.shape) == 3:
            shap_values = shap_values[:, :, 1]
    except Exception:
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]

    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=features,
                      max_display=10, show=True)
    plt.tight_layout()
    plt.show()

    print('> SHAP Summary Plot 解读:')
    print('  红色（高特征值）的 Sex (female=1) 集中在右侧 → 女性强烈正向影响幸存。')
    print('  蓝色（低 Pclass）的 Pclass 集中在右侧 → 高舱位等级正向影响幸存。')
else:
    print('当前模型类型不支持 TreeExplainer，跳过 SHAP 分析。')
    print('对于 Logistic Regression，可直接用系数作为特征重要性。')

## 6.3 业务结论：如果你是泰坦尼克号上的乘客...

基于模型分析，决策幸存的关键因素按重要性排序：

### 🥇 第一决定因素：性别
- 女性和男性之间有 **~55 个百分点**的幸存率差异
- "妇女儿童优先"是当时最严格执行的救生艇分配原则

### 🥈 第二决定因素：舱位等级（财富）
- 头等舱幸存率 **63%**，三等舱仅 **24%**
- 头等舱靠近甲板 + 救生艇，信息获取更早

### 🥉 第三决定因素：年龄
- 儿童（0-12 岁）有明显优势，但**仅限于男孩**
- 成年男性各年龄段幸存率相近（都很低）

### 额外的微妙因素
- **家庭规模 2-4 人最优**：太小（独自）缺乏互助，太大（5+）难以统一行动
- **有 Cabin 记录**的人幸存率更高——不只是甲板位置，更是信息完整性的proxy
- **票价**和 Pclass 高度共线，但在三等舱内部，付更高票价的人幸存率略高

---

### 一句话总结

> **如果你是泰坦尼克号上的成年男性三等舱乘客，你的幸存率不到 10%。**
> **如果你是一个头等舱 8 岁小女孩的母亲，你的幸存率超过 90%。**
>
> **这就是这场灾难最残酷的真相——在生死面前，"阶级 x 性别 x 年龄"的交叉决定了谁上救生艇、谁留在大西洋的冰水里。**

# 第 7 章 · 复盘与改进

## 本节目标

做完不是结束，复盘才是。一个好的分析者在提交结果后会问自己三个问题：**哪里做得不够好？如果再给一轮时间会做什么？这套方法还能用在哪里？**

In [ ]:
# === 7.1 Kaggle 提交 ===
# 预测测试集
predictions = best_model.predict(X_test_final if not any(k in type(best_model).__name__ for k in ['SVM', 'Logistic', 'KNN']) else X_test_scaled)

# 有些模型需要 scaled 数据
if 'Logistic' in type(best_model).__name__ or 'SVC' in type(best_model).__name__ or 'KNN' in type(best_model).__name__:
    predictions = best_model.predict(X_test_scaled)
else:
    predictions = best_model.predict(X_test_final)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})
submission.to_csv('output/submission.csv', index=False)
print(f'提交文件已保存: output/submission.csv ({len(submission)} 条)')

# 检查预测分布
print(f'\n预测幸存: {submission["Survived"].sum()} 人 ({submission["Survived"].mean()*100:.1f}%)')
print(f'预测遇难: {len(submission) - submission["Survived"].sum()} 人')
print(f'\n训练集幸存率: {train["Survived"].mean()*100:.1f}% —— 预测分布与训练集基本一致 ✓')

## 7.2 复盘：当前方案的不足

1. **Name 中的更多信息未充分挖掘**
   - 目前只提取了 Title。其实 Name 中还有姓氏（家族分组）、中间名（可能包含娘家姓），这些都可以构造"是否大家族成员"等特征

2. **Ticket 的前缀信息未利用**
   - Ticket 列有很多带有字母前缀的票号（如 `PC 17599`），前缀可能代表售票代理或航线，和 Embarked 有关

3. **特征交互靠手工，缺少自动化**
   - Sex×Pclass、Age×Pclass 这些交互是我在 EDA 中手工发现的。如果特征更多，应该用特征选择算法或交互特征自动生成

4. **模型集成不够多样**
   - 没有尝试 Stacking（多模型投票）或 Voting Classifier。对于提升最后 1-2% 的准确率，集成比调参更有效

## 7.3 如果有第二轮，我会做什么

| 优先级 | 方向 | 预期收益 |
|--------|------|----------|
| ⭐⭐⭐ | Stacking Ensemble（LR + RF + XGBoost） | +1~2% accuracy |
| ⭐⭐ | 更深入的 Name 特征（姓氏分组、双名检测） | +0.5~1% |
| ⭐⭐ | Bayesian 优化替代 GridSearch | 更快且更优的参数 |
| ⭐ | Ticket 前缀特征 | 信息量存疑，值得验证 |

## 7.4 从 Titanic 到真实场景

这个分析框架可以直接迁移到很多**二分类问题**：

| Titanic 概念 | 可迁移场景 |
|-------------|-----------|
| 预测某人是否幸存 | 客户是否流失、贷款是否违约、用户是否转化 |
| Sex/Pclass 是核心特征 | 寻找业务中的"第一性因素" |
| EDA 先画图再建模 | 任何数据分析的第一步 |
| 缺失值逐个分析处理 | 真实数据永远不干净 |
| SHAP 解释模型 | 让业务方信任你的模型 |

> **数据分析的核心不是写代码，是"发现问题 → 提出假设 → 数据验证 → 得出结论"这个思维闭环。Titanic 是练这个闭环的最好沙盘。**